In [1]:
%%html
<style>
    h1 {color:darkblue}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>


# Outline
0. Part 0 — Intro/Setup
1. **Part 1 — OpenAI Python APIs**
   * 01-01. **Text Generation Via the Responses API**
   * 01-02. Speech Recognition, Speech Synthesis, and Closed Captions
   * 01-03. Images: Generation and Style Transfer
   * 01-04. Content Moderation
   * 01-05. Generating Code with a Codex Model and the Responses API
2. Part 2 — OpenAI Agents SDK
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Text Generation Via the Responses API
* Apps that interact with OpenAI's GPT models to generate text responses based on text prompts

## Examples/Techniques this notebook:
* Shared setup: imports, API key assumption, and one `OpenAI` client object
* Summarizing a transcript and extracting key points
    * Rendering Markdown responses and streaming generated text
* **\[Python Fundamentals\]** Sentiment analysis of a presentation transcript
* Vision: Accessible image descriptions from image inputs
* Language detection and translation
* Named entity recognition (NER) with structured outputs
    * Three structured-output approaches
        * Prompt-only JSON
        * Strict JSON Schema
        * Pydantic typed objects
---


## Nondeterministic Responses
* Responses returned by genAIs are nondeterministic
    * Typically change from request to request
    * Even if your prompts are identical
    * Even when submitted to the same genAI
* Could be problematic for some applications
---

# Creating the `OpenAI` Client Object

In [2]:
from openai import OpenAI 

In [3]:
client = OpenAI()

* Assumes API key is in `OPENAI_API_KEY` environment variable

* If you used a different environment variable name, replace with
>```python
> client = OpenAI(api_key=os.environ.get("CUSTOM_ENV_NAME"))
>```

* If you store your key elsewhere, such as in a file
>```python
> import keys
> client = OpenAI(api_key=keys.OPENAI_API_KEY)
>```
---

# Common Imports for This Notebook

In [4]:
import json
from pathlib import Path
from IPython.display import display, Markdown                              

--- 

# Example 1 — Text Summarization and Streaming Responses
* Analyze a body of text and produce a concise version capturing key ideas
    * Quickly summarize articles, academic papers, book chapters or sections, emails, reports, study materials, product reviews, …
* Summarize a transcript as an abstract paragraph.
* Extract key points.
* Render Markdown.
* Stream a combined response as it is generated.
* `transcript.txt` (located in the `resources` subfolder) 
    * Created from introduction to Lesson 1's Test-Drives from my **Python Fundamentals** video course

### Loading the Presentation Transcript


In [5]:
transcript_path = Path('resources') / 'transcript.txt'

In [6]:
transcript = transcript_path.read_text()

In [7]:
print(transcript)

Welcome to lesson one, Test Driving IPython and Jupyter Notebooks. In this lesson, we'll be looking at several software test drives, and we will begin specifically with the IPython interpreter and what's known as its interactive mode. With the interactive mode of IPython, we're going to be able to enter small bits of Python code, which we'll refer to as snippets, and Python will then read the snippet we enter, evaluate the snippet we enter. If that snippet has a result associated with it, it will print the result, so we'll immediately see that result, and then it will loop and wait for the next snippet to be entered by us. Read, evaluate, print, and loop together is known as a REPL environment, or R-E-P-L for short. So this is a great way to learn Python because you don't need to write full programs to practice the capabilities of the language, and it's a frequent mode for people who are doing things like exploratory data analysis where they may not know in advance everything they want

### Requesting a Response That Produces a Summary Abstract Paragraph


In [8]:
model_instructions = """Given a Python technical presentation's
transcript, create a summary abstract paragraph. Use straightforward
sentences. Spell language features and function names correctly. 
Avoid abbreviations. Do not refer to the speaker."""

In [9]:
response = client.responses.create(model='gpt-5.4-nano', 
    instructions=model_instructions, input=transcript)

### Displaying the Response


In [10]:
print(response.output_text)

Lesson one introduces test driving IPython and Jupyter Notebooks. It starts with the IPython interpreter in interactive mode, where short code snippets are read, evaluated, and printed in a repeatable REPL environment that supports exploratory data analysis without writing full programs. Next it covers IPython script mode, which runs complete Python scripts stored in .py files and can define reusable functions and classes, illustrated with a die-rolling example that generates a bar plot of simulated frequencies. The lesson then turns to Jupyter Notebooks, which are document files with the IPYNB extension that combine rich text, media, and executable Python code, and can also support other languages through multiple Jupyter kernels. It highlights JupyterLab as the modern IDE-like interface for editing and executing notebooks, explains how notebooks package analysis with explanatory text and results, and concludes with a look at Google CoLab as a cloud service for running compatible IPYN

### Token Counting and Cost Awareness
* Cost is based on input tokens and output tokens
* `response.usage` contains this info
* [Model pricing details](https://openai.com/api/pricing/ )
* [Your Usage Dashboard @ https://platform.openai.com/usage](https://platform.openai.com/usage)

In [11]:
print(f'{response.usage.input_tokens=}\n{response.usage.output_tokens=}')

response.usage.input_tokens=1232
response.usage.output_tokens=194


### Other GPT Models
* `'gpt-5.5'`
  * Current most powerful "thinking" model
* `'gpt-5.4-mini'`
  * Faster, cheaper; some "thinking"/planning before responding
* All `gpt-5.x` models can generate text, images and code

### Extracting Key Points


In [12]:
model_instructions = """Given a Python technical presentation's
transcript, present a numbered list of the top 5 key points. 
Use concise, direct sentences and avoid abbreviations."""

In [13]:
response = client.responses.create(model='gpt-5.4-nano', 
    instructions=model_instructions, input=transcript)

In [14]:
print(response.output_text)

1. IPython interactive mode runs small Python snippets in a REPL loop that reads, evaluates, prints results, and waits for the next input.  
2. REPL usage supports exploratory work, such as exploratory data analysis and incremental experimentation without writing full programs.  
3. IPython script mode runs complete Python programs from script files, which can also define functions and classes as reusable libraries.  
4. Jupyter Notebooks are shareable documents with executable Python code and rich content like text, images, and media, using an IPYNB-based format.  
5. JupyterLab is an IDE-like environment for developing and running notebooks, and Google CoLab offers cloud-based notebook execution with compatibility for IPYNB files.


### Markdown-Formatted Responses


In [15]:
Markdown(response.output_text)

1. IPython interactive mode runs small Python snippets in a REPL loop that reads, evaluates, prints results, and waits for the next input.  
2. REPL usage supports exploratory work, such as exploratory data analysis and incremental experimentation without writing full programs.  
3. IPython script mode runs complete Python programs from script files, which can also define functions and classes as reusable libraries.  
4. Jupyter Notebooks are shareable documents with executable Python code and rich content like text, images, and media, using an IPYNB-based format.  
5. JupyterLab is an IDE-like environment for developing and running notebooks, and Google CoLab offers cloud-based notebook execution with compatibility for IPYNB files.

## Streaming a Response

* Preceding calls wait for **complete response**
* `client.responses.create` 
    * `stream=True` to return an **event stream** of incremental updates
    * Like chatbots streaming their responses 
* `ResponseTextDeltaEvent.delta` 
    * Small text chunk rather than full `.output_text`
* Accumulate deltas into a string 
* `IPython.display.display(..., display_id=True)`
    * Returns handle for **updating a single output cell** rather than printing many separate lines
* `output_handle.update(Markdown(...))`
    * Re-render same output area in place as each chunk arrives

### Streaming a Combined Summary and Key Points Response


In [16]:
model_instructions = """Given a Python technical presentation's transcript:

1. Write a summary abstract paragraph using straightforward sentences.
   Spell language features and function names correctly. Avoid
   abbreviations. Do not refer to the speaker.

2. Then present a numbered list of the top 5 key points using concise,
   direct sentences. Avoid abbreviations.

Format your response using Markdown with clear section headings."""

In [18]:
accumulated_text = ''
output_handle = display(Markdown(''), display_id=True)

with client.responses.create(model='gpt-5.4-nano',
        instructions=model_instructions, input=transcript,
        stream=True) as stream:
    for event in stream:
        if event.type == 'response.output_text.delta':
            accumulated_text += event.delta
            output_handle.update(Markdown(accumulated_text))

## Summary Abstract

This lesson introduces three ways to run Python code: the IPython interpreter in interactive mode, IPython script mode, and Jupyter Notebooks. In interactive mode, the IPython interpreter reads small code snippets, evaluates them, prints any resulting output, and then waits for the next snippet, which creates a REPL environment. Script mode runs complete Python programs stored in `.py` files and allows those files to provide reusable code such as function definitions and class definitions. The lesson also explains Jupyter Notebooks as document-based workflows that combine rich text, media, and executable Python code stored in `.ipynb` files. It describes JupyterLab as a modern IDE-like interface for writing and executing notebook content, and it briefly covers Google Colab as a cloud service for running Jupyter Notebooks without local installation.

## Top 5 Key Points

1. IPython interactive mode evaluates short Python snippets in a REPL loop.  
2. IPython script mode runs full Python scripts from `.py` files.  
3. Python scripts can define reusable functions and classes for later use.  
4. Jupyter Notebooks store executable code with rich text in `.ipynb` documents.  
5. JupyterLab and Google Colab provide notebook interfaces that support interactive execution, including on cloud infrastructure.

---

# Example 2 — Sentiment Analysis
* Reuse transcript loaded in the summarization example and ask the model to classify and explain the sentiment
    * Positive, neutral or negative

In [ ]:
model_instructions = """You are a sentiment-analysis expert. 
Determine the provided transcript's sentiment. Explain your analysis."""

In [ ]:
response = client.responses.create(model='gpt-5.4-mini', 
    instructions=model_instructions, input=transcript)

In [ ]:
Markdown(response.output_text)

---
# Example 3 — Accessible Image Descriptions
* Responses API and vision capabilities
* World Wide Web Consortium (W3C): 
    * Web should be accessible to everyone, regardless of ability
* Accessible image descriptions
    * For people who are blind or have low vision
* Multimodal application with image and text inputs
  * Uploads an image
  * Requests a detailed image description based on the World Wide Web Consortium (W3C) accessible-image guidelines

---

## Import Function `describe_image`
* [Open describe_image.py](./source/describe_image.py)
* [Open util.py](./source/util.py)

In [19]:
from source.describe_image import describe_image

--- 

### Getting an Accessible Description of a Scenic Aruba Sunset Photo
<img src="./resources/sunset.jpg" width="75%"/>


In [20]:
description = describe_image(client,
    'Provide an accessible description of this Aruba sunset.', 
    Path('resources') / 'sunset.jpg')

In [21]:
Markdown(description)

A vivid sunset over the ocean in Aruba. The sky fills most of the image with glowing orange and gold light, fading to softer tan and gray near the top. Dark, scattered clouds drift across the scene, including a large cloud partly covering the sun near the horizon. The sun is low and bright, creating a strong golden reflection that stretches down the rippling water toward the foreground. The sea appears dark blue-gray, with a thin, straight horizon line separating it from the sky. Near the bottom edge, a small buoy or floating marker is visible in the water, and a few dark posts or railings silhouette the shoreline. The overall mood is warm, calm, and dramatic.

---

### Getting an Accessible Description of a Word Cloud
<img src="./resources/RomeoAndJulietHeart.png" width="75%"/>


In [ ]:
description = describe_image(client,
    """Provide an accessible description of this 
    Romeo and Juliet word cloud.""", 
    Path('resources') / 'RomeoAndJulietHeart.png')

In [ ]:
Markdown(description)

## Creating the Local Image File's URL
* Images are stored as bytes
* When transmitted over the Internet between systems that handle the raw bytes differently, the bytes can get corrupted
* Those systems typically understand characters
* Raw bytes often encoded into character representations
* Base64 — named for the Base64 alphabet's 64 characters
  * 52 letters (`A`–`Z` and `a`–`z`)
  * 10 digits (`0`–`9`)
  * `+` and `/`
* All printable, easily understood by world's computer systems
---

## Data URL Format
* Local images passed to Responses API must be Base64-encoded and embedded in a URL with the format
  * `data:image/imageFormat;base64,base64EncodedImageString`
    * `imageFormat` in our example is `png` or `jpeg`
    * `image/imageFormat` is the MIME type of the image
* `create_data_url` utility function (in [util.py](./source/util.py))
  * Base64-encodes the image
  * Returns a properly formatted `data:` URL
---

## Passing the Image to the Responses API
* [Open describe_image.py](./source/describe_image.py)
* Responses API multimodal inputs use different input format
* API requires JSON array of JSON objects, each with an optional text prompt and the URL of an image to analyze
* In Python, Multimodal inputs require a list of dictionaries
  * Auto formatted by `responses.create` as JSON
  * We analyze only one image at a time, so the list in this example contains one dictionary

---

## Accessible Image Descriptions — Dictionary Format

* Each dictionary requires two key-value pairs:
  * `'role': 'user'` — input includes the user's data on which model operates
  * `'content'` key's value — list of data to provide to model as input
  * When uploading an image, the `'content'` key's value is a list of dictionaries
* Optional: Prompt dictionary with context or instructions
  * `'type': 'input_text'`
  * `'text'` key with the prompt string as its value
* Image dictionary
  * `'type': 'input_image'`
  * `'image_url'` — `data:` URL containing Base64-encoded image

---
# Example 4 — Language Detection and Translation
* genAIs can translate between languages
* Wide range of uses, including:
  * Cross-language communication: Facilitating conversations between people speaking different languages
  * Global business operations: Translating documents, websites and customer communications for international markets
  * Tourism: Helping travelers communicate in foreign countries
  * Education: Enabling students and researchers to access academic content in different languages
  * Healthcare: Assisting healthcare providers in understanding patients who speak different languages
  * Diplomatic relations: Supporting negotiations and communications between countries, such as at the United Nations
  * Entertainment: Translating subtitles, scripts and media for global audiences

## Translation Example
* Use Responses API to translate text among languages
* Autodetect the text's source language then translate the text to a specified target language
  * Translate English to Spanish and Japanese
  * Translate the Spanish and Japanese text back to English
* Well-defined task that does not require significant reasoning, so we’ll use gpt-5.4-mini

## Translate English to Spanish and Japanese
* Automatically figures out source language
* [Open translate.py](./source/translate.py)

In [22]:
from source.translate import translate

In [23]:
english_text = (
    'Today was a beautiful day. Tomorrow looks like bad weather.')

In [24]:
spanish_text = translate(client, english_text, 'Spanish')
spanish_text

'Hoy fue un día hermoso. Mañana parece que hará mal tiempo.'

In [25]:
japanese_text = translate(client, english_text, 'Japanese')
japanese_text

'今日は美しい日でした。明日は悪天候になりそうです。'

### Saving Translated Text for the Speech Notebook
* Save in `resources/outputs` so `01-02-speech.ipynb` can reuse them later

In [26]:
translated_text_dir = Path('resources') / 'outputs'
translated_text_dir.mkdir(parents=True, exist_ok=True)

spanish_text_path = translated_text_dir / 'spanish_text.txt'
japanese_text_path = translated_text_dir / 'japanese_text.txt'

spanish_text_path.write_text(spanish_text, encoding='utf-8')
japanese_text_path.write_text(japanese_text, encoding='utf-8')

spanish_text_path, japanese_text_path

(PosixPath('resources/outputs/spanish_text.txt'),
 PosixPath('resources/outputs/japanese_text.txt'))

### Translating Spanish and Japanese Back to English


In [27]:
translate(client, spanish_text, 'English')

'Today was a beautiful day. Tomorrow it looks like the weather will be bad.'

In [28]:
translate(client, japanese_text, 'English')

'Today was a beautiful day. Tomorrow seems likely to have bad weather.'

In [29]:
english_text

'Today was a beautiful day. Tomorrow looks like bad weather.'

---
# Example 5 — Named Entity Recognition (NER) and Structured Outputs
* Locate and categorize entities such as dates, times, quantities, places, people, things, organizations and more
* Structured JSON output
    * Make results easier to process programmatically
    * We'll provide the exact JSON response format we want
    * Use the json module to process and display the results

---

### NER Tag Set
* Initial tests gave different results for each run
  * Sometimes using tags
  * Sometimes using words
* For more deterministic results, told model to use OntoNotes Named Entity Tag Set
  * spaCy uses this in its NER results

---

### JSON Output for Named Entities
* Want model to return response as a JSON object with an `"entities"` key and an array of JSON objects as its value
* Each object in the array contains:
  * `"text"` attribute for a named entity
  * `"tag"` attribute indicating that named entity's type
* Include this as part of request

```json
{
   "entities": [
      {
         "text": "Entity name",
         "tag": "Entity type"
      }
   ]
}
```

---


### Loading the Text
* We'll perform NER on the text in the file web.txt, which contains a paragraph about the World Wide Web from Chapter 1
* This file is located in the resources subfolder with this chapter's examples


In [30]:
path = Path('resources') / 'web.txt'

In [31]:
text = path.read_text()

In [32]:
text

'The World Wide Web (simply called "the web") is a collection of hardware and software associated with the Internet that allows computer users to locate and view documents (with various combinations of text, graphics, animations, audios and videos) on almost any subject. In 1989, Tim Berners-Lee of CERN (the European Organization for Nuclear Research) began developing HyperText Markup Language (HTML)—the technology for sharing information via "hyperlinked" text documents. He also wrote communication protocols such as HyperText Transfer Protocol (HTTP) to form the backbone of his new hypertext information system, which he referred to as the World Wide Web. In 1994, Berners-Lee founded the World Wide Web Consortium (W3C, https://www.w3.org), which was devoted to developing web technologies. A key W3C goal is to make the web universally accessible to everyone regardless of disabilities, language or culture.'

---

### Performing NER on the Loaded Text
* Submit a request to the Responses API containing:
  * Model to use (`'gpt-5.4-mini'`)
  * Instructions telling the model it's an expert in named entity recognition using the OntoNotes Named Entity Tag Set
  * Input prompt that asks the model to analyze the text's named entities and return them in JSON format shown in the prompt


In [33]:
response = client.responses.create(
    model='gpt-5.4-mini',
    instructions="""You are an expert in named entity 
        recognition with the OntoNotes Named Entity Tag Set.""",
    input="""Analyze the supplied text and extract its named
        entities. Return only JSON in the following format:
            {
              "entities": [
                {
                  "text": "Entity name",
                  "tag": "Entity type"
                }
              ]
            }   
        Text to analyze:
        """ + text
)

### Deserializing the response
* Use the `json` module's `loads` ("load string") function to deserialize the JSON in its string argument, creating the dictionary json_response
* For readability, use the `json` module's `dumps` ("dump string") function to create a structured string representation of the named entities
* The output shows the named entities categorized by type 

In [34]:
json_response = json.loads(response.output_text)

In [35]:
print(json.dumps(json_response, indent=2))

{
  "entities": [
    {
      "text": "World Wide Web",
      "tag": "ORG"
    },
    {
      "text": "Internet",
      "tag": "ORG"
    },
    {
      "text": "1989",
      "tag": "DATE"
    },
    {
      "text": "Tim Berners-Lee",
      "tag": "PERSON"
    },
    {
      "text": "CERN",
      "tag": "ORG"
    },
    {
      "text": "European Organization for Nuclear Research",
      "tag": "ORG"
    },
    {
      "text": "HyperText Markup Language",
      "tag": "WORK_OF_ART"
    },
    {
      "text": "HTML",
      "tag": "WORK_OF_ART"
    },
    {
      "text": "HyperText Transfer Protocol",
      "tag": "WORK_OF_ART"
    },
    {
      "text": "HTTP",
      "tag": "WORK_OF_ART"
    },
    {
      "text": "World Wide Web",
      "tag": "ORG"
    },
    {
      "text": "1994",
      "tag": "DATE"
    },
    {
      "text": "Berners-Lee",
      "tag": "PERSON"
    },
    {
      "text": "World Wide Web Consortium",
      "tag": "ORG"
    },
    {
      "text": "W3C",
      "tag": "

## Three Approaches to Structured Responses
* Approach 1 — Prompt-Based (No Schema Enforcement) 
* Approach 2 — Raw JSON Schema (API-Enforced)
* Approach 3 — Pydantic (Typed Python Objects)
---


### Approach 1 — Prompt-Based (No Schema Enforcement)
This is what we just did.

**Pros**
* Zero setup — describe the format in `instructions` or `input` string
* Works with any model; no extra SDK parameters needed
* Easy to adjust the schema by editing a string

**Cons**
* No enforcement — the model may deviate from the requested shape or return malformed JSON
* `json.loads()` can raise `JSONDecodeError` at runtime if the model misbehaves
* Extra or missing fields cause bugs 
* All validation is your responsibility

---


### Approach 2 — Raw JSON Schema (API-Enforced)

**Pros**
* API guarantees response matches schema — `json.loads()` always succeeds
* Schema is explicit and reusable
* Ideal when schema is constructed dynamically at runtime or read from an external source

**Cons**
* Writing JSON schema by hand is error-prone
* Result is still a plain `dict` and `list` objects 
    * no Python type checking or IDE autocomplete
* `additionalProperties: false` and `required` arrays must be set on every object
    * Otherwise, OpenAI API rejects the schema
---

#### JSON Schema


In [ ]:
ner_schema = {
    'type': 'object',
    'properties': {
        'entities': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'text': {'type': 'string'},
                    'tag':  {'type': 'string'}
                },
                'required': ['text', 'tag'],
                'additionalProperties': False
            }
        }
    },
    'required': ['entities'],
    'additionalProperties': False
}

```python
ner_schema = {
    # top-level output must be a JSON object: { ... }
    'type': 'object',

    # object may contain these named properties
    'properties': {
        # nne property named "entities"
        'entities': {
            # "entities" must be an array/list
            'type': 'array',

            # each array item must follow this schema
            'items': {
                # each array item must be an object
                'type': 'object',

                # each object may contain these properties
                'properties': {
                    # actual text found in the input
                    'text': {'type': 'string'},

                    # entity label/category
                    'tag':  {'type': 'string'}
                },

                # every object must include both fields
                'required': ['text', 'tag'],

                # no extra fields allowed inside each entity object
                'additionalProperties': False
            }
        }
    },

    # top-level object must include "entities"
    'required': ['entities'],

    # no extra top-level fields allowed
    'additionalProperties': False
}
```

#### Calling `responses.create` with `text={'format': ...}`


In [ ]:
response = client.responses.create(
    model='gpt-5.4-mini',
    instructions="""You are an expert in named entity
        recognition with the OntoNotes Named Entity Tag Set.""",
    input='Analyze the supplied text and extract its named entities.\n\n' + text,
    text={
        'format': {
            'type': 'json_schema',
            'name': 'NERResult',
            'schema': ner_schema,
            'strict': True # output must confirm to this schema
        }
    }
)

#### Deserializing — Always Succeeds Because the API/Model Enforce the Schema

In [ ]:
json_response = json.loads(response.output_text)
print(json.dumps(json_response, indent=2))

---

### Approach 3 — Pydantic (Typed Python Objects)
* https://docs.pydantic.dev
* Structured data validation 
* Pydantic enforces types and validates data at runtime using standard Python type hints

#### Pros for Structured Outputs
* Define the schema once as Pydantic `BaseModel` 
* SDK generates JSON Schema automatically from Pydantic model
* `response.output_parsed` is a **typed Python object** — no `json.loads()` needed
* IDE autocomplete and static type checking support
* **Best for production code**

#### **Cons** for Structured Outputs
* Requires Pydantic (but this is already an OpenAI SDK dependency)
* Model must be defined ahead of time — less flexible for truly dynamic schemas
* More initial setup

### Defining a Pydantic Model
* `BaseModel`
    * Superclass for defining a schema
    * Fields declared as annotated class attributes

In [36]:
from pydantic import BaseModel
from typing import Optional

class Person(BaseModel):
    name: str
    age: int
    email: Optional[str] = None

* `pydantic.Field(...)` can be used to add descriptive metadata to fields
    * `description=`, `default=`, `gt=`, `lt=`, ...
    * `description` becomes part of JSON schema model uses when generating structured output

##### Pydantic validates and coerces types on construction

In [37]:
p1 = Person(name="Alice", age="30")   # age coerced from str to int 

In [38]:
p1.age

30

In [39]:
p2 = Person(name="Alice", age="abc") # raises ValidationError as "abc" is not convertable to int

ValidationError: 1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='abc', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

#### JSON for a Pydantic Object and Schema
* `model_object.model_dump()` 
    * Returns a `dict` representation of the schema
* `model_object.model_json_schema()` 
    * Returns the full JSON schema

In [40]:
p1.model_dump()

{'name': 'Alice', 'age': 30, 'email': None}

In [41]:
print(json.dumps(p1.model_json_schema(), indent=2))

{
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "age": {
      "title": "Age",
      "type": "integer"
    },
    "email": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Email"
    }
  },
  "required": [
    "name",
    "age"
  ],
  "title": "Person",
  "type": "object"
}


#### Pydantic and the Responses API
* Pass a Pydantic model as `text_format=` to `client.responses.parse()`
    * SDK serializes it to JSON schema
    * SDK sends schema to model as a constraint
    * SDK deserializes model's JSON output into a validated Python object
        * `response.output_parsed`

#### Defining the Pydantic Models


In [42]:
from pydantic import BaseModel 

class Entity(BaseModel):
    text: str
    tag: str

class NERResult(BaseModel):
    entities: list[Entity]

#### Calling `responses.parse` with `text_format=NERResult`

* Use `client.responses.parse` instead of `client.responses.create`
* Pass the Pydantic class as `text_format` — the SDK converts it to JSON Schema for you
* `response.output_parsed` is an `NERResult` instance — no `json.loads()` required


In [43]:
response = client.responses.parse(
    model='gpt-5.4-mini',
    instructions="""You are an expert in named entity
        recognition with the OntoNotes Named Entity Tag Set.""",
    input='Analyze the supplied text and extract its named entities.\n\n' + text,
    text_format=NERResult 
)

#### Accessing the Typed Result


In [44]:
result: NERResult = response.output_parsed

for entity in result.entities:
    print(f'{entity.tag:20} {entity.text}')

ORG                  the World Wide Web
ORG                  Internet
DATE                 1989
PERSON               Tim Berners-Lee
ORG                  CERN
ORG                  the European Organization for Nuclear Research
ORG                  HyperText Markup Language
ORG                  HTML
ORG                  HyperText Transfer Protocol
ORG                  HTTP
ORG                  the World Wide Web
DATE                 1994
PERSON               Berners-Lee
ORG                  World Wide Web Consortium
ORG                  W3C
ORG                  https://www.w3.org


---

### Note: `output_parsed` is typed `Optional[ContentType]`
* Can return `None` if response contains no parsed output message
* Typically, not `None` when `text_format` is passed
* Defensive programming: Check that `result` is not `None`

---

### Summary of Pros/Cons 

| | Prompt-Based | Raw JSON Schema | Pydantic |
|---|---|---|---|
| **Schema enforced by API** | No | Yes | Yes |
| **Python type safety** | No | No | Yes |
| **`json.loads()` required** | Yes — can raise `JSONDecodeError` | Yes — always succeeds | No |
| **IDE autocomplete on result** | No | No | Yes |
| **Dynamic / runtime schemas** | Yes | Yes | Harder |
| **Best for** | Quick prototypes | External/dynamic schemas | Production code |

---

# Additional Info for Your Reference
## Error Handling — `RateLimitError`, `APIStatusError`, and Retries

* OpenAI Python SDK exceptions inherit from `openai.APIError`
* **`RateLimitError`** maps to HTTP `429`
    * Often means too many requests or tokens in a rate-limit interval
    * Back off and retry; avoid retrying indefinitely
* **`APIStatusError`**
    * Raised for non-success HTTP responses (`4xx` or `5xx`)
    * Exposes `.status_code`, `.response`, and `.request_id`
    * Use `str(e)` for the error message
* **`max_retries`**
    * Configures automatic short exponential-backoff retries
    * Default is `2`
    * Retries include connection errors, `408`, `409`, `429`, and `>=500`
    >```python
    ># configure automatic retry count 
    >client = openai.OpenAI(max_retries=5)
    >```

### Reference Links
* **Error codes guide:** https://developers.openai.com/api/docs/guides/error-codes
* **Python library — Handling errors:** https://github.com/openai/openai-python?tab=readme-ov-file#handling-errors
* **Python library — Retries:** https://github.com/openai/openai-python?tab=readme-ov-file#retries
* **Rate limits guide:** https://platform.openai.com/docs/guides/rate-limits

---

## `client.responses.create` — Common Parameters
| Parameter | Purpose |
|---|---|
| `model` | Selects the model to use |
| `instructions` | Sets system/developer-level behavior |
| `input` | Supplies the user prompt or input items |
| `max_output_tokens` | Caps generated output length and helps control cost |
| `temperature` | Adjusts output randomness/variety |
| `text` | Controls text output format, including structured JSON |

### Reference Link
* [Responses API — Create a model response](https://developers.openai.com/api/reference/resources/responses/methods/create)  


<hr/>
© 2026 by Deitel & Associates, Inc. All Rights Reserved.